<p style="text-align: center; margin-bottom: 0; font-size: 24px;">
EAE1106 - Métodos Computacionais para Economia
</p>

<p style="text-align: center; margin-bottom: 0;">
Faculdade de Economia, Administração, Contabilidade e Atuária<br>
Universidade de São Paulo
</p>
<p style="text-align: center; margin-top: 5px;">
1º semestre de 2026<br><br>
Prof. Arthur Viaro
</p>

---

# Aula 10 - Funções: Continuação

- [Docstrings](#docstrings)
- [Escopo de variáveis](#escopo)
- [args e kwargs](#args-e-kwargs)
- [Funções anônimas](#funcoes-anonimas)

## Docstrings <a id="docstrings"></a>

**Docstrings** são strings de documentação que descrevem o que uma função faz, quais parâmetros ela recebe e o que ela retorna. São escritas logo abaixo do cabeçalho da função, entre aspas triplas (`"""`).

Boas docstrings tornam o código muito mais fácil de entender e de manter — especialmente quando você retorna a um código depois de semanas ou quando compartilha com colegas. Você pode acessar a docstring de qualquer função com `help()` ou `função.__doc__`.

O formato mais comum em Python é o **Google Style** ou o **NumPy Style**.

In [ ]:
def taxa_desemprego(desempregados, populacao_ativa):
    """
    Calcula a taxa de desemprego de uma economia.

    Parâmetros:
        desempregados (float): número de pessoas desempregadas.
        populacao_ativa (float): tamanho da força de trabalho (PEA).

    Retorna:
        float: taxa de desemprego, expressa em percentual (%).
    """
    return (desempregados / populacao_ativa) * 100

help(taxa_desemprego) # Consultando a documentação da função

In [ ]:
# Dados hipotéticos (em milhões de pessoas)
desempregados = 8.5
pea = 107.0

taxa = taxa_desemprego(desempregados, pea)
print(f"Taxa de desemprego: {taxa:.1f}%")

## Escopo de variáveis <a id="escopo"></a>

- O escopo de uma variável se refere ao local onde uma variável é definida e onde ela pode ser acessada.
- Os nomes definidos dentro de um escopo são armazenados em um objeto chamado `namespace`. Ele atua como um dicionário, onde a chave é o nome (ou rótulo) de um objeto e o valor é o objeto referenciado.
- Em Python, existem quatro tipos de escopos: *local*, *enclosing*, *global* e *built-in*.
- Lembre-se, Python segue a regra **LEGB** para resolver os nomes das variáveis. Primeiro, verifica o escopo Local, depois o Enclosing, depois o Global e finalmente o Built-in.

**Escopo Local**: Uma variável definida dentro de uma função é chamada de variável local. Ela só pode ser acessada dentro dessa função.

In [ ]:
def funcao_local():
  mensagem = "Olá, Mundo!"  # Variável do escopo local
  print(mensagem)

funcao_local()  # Saída: Olá, Mundo!
print(mensagem) # Erro: mensagem não está definida fora da função

**Escopo Enclosing**: Se uma função está dentro de outra função (função aninhada), a função interna tem acesso às variáveis da função externa. Este é o escopo enclosing.

- No exemplo a seguir, a função interna consegue acessar a variável `mensagem` da `funcao_externa`, que está no escopo Enclosing.
- Isso acontece exatamente por causa da regra LEGB: o Python busca os nomes de dentro para fora.
- Por outro lado, a `funcao_externa` não tem acesso direto à variável `mensagem`, pois ela reside em um escopo mais interno que o dela.
- Da mesma forma, o escopo global não consegue acessar nenhuma das variáveis definidas dentro de `funcao_externa` ou `funcao_interna`

In [ ]:
def funcao_externa():
  mensagem = "Olá, Mundo!" # Variável da função externa

  def funcao_interna():
    print(mensagem) # Acesso à variável da função externa

  funcao_interna() # Chamamos a função interna para executá-la

funcao_externa()

**Escopo Global**: Uma variável definida fora de todas as funções é uma variável global e pode ser acessada em qualquer lugar do programa.

In [ ]:
poupanca = 1000  # Variável no escopo global

def banco():
  print(f"Saldo atual: R${poupanca}")  # Acessa 'poupanca' do escopo global

banco()

No exemplo anterior, quando `print()` é chamado dentro da função `banco()`, o Python primeiramente busca a variável poupanca no escopo local da função. Como não o encontra lá, ele "sobe" para o escopo global, onde a variável está disponível. Note, porém, o que acontece se tentarmos alterar o valor da variável poupança:

In [ ]:
poupanca = 1000

def deposito(n):
    poupanca += n  # Erro: UnboundLocalError

def retirada(n):
    poupanca -= n  # Erro: UnboundLocalError

def banco():
  deposito(500)
  retirada(200)
  print(f"Saldo atual: R${poupanca}")

banco()

Observe que adicionamos a funcionalidade de depositar e sacar valores do saldo. No entanto, ao executar o código, nos deparamos com um erro chamado `UnboundLocalError`. Isso acontece porque o Python, ao encontrar `poupanca += n` dentro de `deposito`, entende que `poupanca` é uma variável *local* da função (já que está sendo atribuída). Mas como ela ainda não tem valor local, o Python não sabe de onde pegá-la.

Para modificar uma variável global dentro de uma função, é necessário utilizar a palavra-chave `global`.

In [ ]:
poupanca = 1000

def deposito(n):
    global poupanca
    poupanca += n

def retirada(n):
    global poupanca
    poupanca -= n

def banco():
    deposito(500)
    retirada(200)
    print(f"Saldo atual: R${poupanca}")

banco()

> ⚠️ **Boa prática:** embora `global` resolva o problema, seu uso excessivo dificulta a leitura e a manutenção do código — fica difícil rastrear onde uma variável está sendo alterada. Em programas maiores, é preferível **passar o valor como parâmetro e retorná-lo explicitamente**. Você aprenderá formas mais robustas de gerenciar estado à medida que avançar no curso.

- A palavra-chave `nonlocal` tem uma função similar a `global`, mas atua no escopo Enclosing, ou seja, em variáveis de funções externas que envolvem a função atual, mas que não são o escopo global.
- Isso é particularmente útil quando uma função interna precisa modificar uma variável da função que a contém.
- No exemplo abaixo, `registrar_transacao` mantém um contador de operações internamente, e a função interna `transacao` precisa de `nonlocal` para incrementá-lo:

In [ ]:
def registrar_transacoes():
    total_transacoes = 0  # variável da função externa

    def transacao(descricao):
        nonlocal total_transacoes   # acessa e modifica a variável externa
        total_transacoes += 1
        print(f"  Transação {total_transacoes}: {descricao}")

    transacao("Depósito de R$ 500")
    transacao("Pagamento de boleto R$ 120")
    transacao("Transferência de R$ 200")
    print(f"Total de transações registradas: {total_transacoes}")

registrar_transacoes()

**Escopo Built-in**: São nomes pré-definidos no módulo builtins do Python. Eles são sempre acessíveis.

In [ ]:
print(len('Python'))  # len é uma função built-in

## args e kwargs <a id="args-e-kwargs"></a>

- Em Python, `*args` e `**kwargs` são formas flexíveis de passar múltiplos argumentos para uma função, permitindo lidar com quantidades variáveis de dados — algo bastante útil em modelos econômicos, onde o número de variáveis pode mudar.

- Por exemplo, observe a documentação da função `print` que vimos anteriormente neste curso:

```python
print(*objects, sep=' ', end='\n', file=sys.stdout, flush=False)
```

- `args` são argumentos posicionais, como em `print("Hello", "World")`.
- `kwargs` são argumentos nomeados (ou "keyword arguments"), como em `print(end="")`.
- Como vemos na documentação da função acima, podemos definir funções que aceitam uma quantidade indefinida de argumentos posicionais e/ou nomeados, tornando-as muito mais flexíveis e adaptáveis a diferentes necessidades.

> 💡 **Importante:** `args` e `kwargs` são apenas **convenções de nomenclatura**, não palavras reservadas do Python. O que realmente importa são os asteriscos: `*` para coletar múltiplos argumentos posicionais e `**` para múltiplos nomeados. Você poderia escrever `*numeros` ou `**indicadores` e o comportamento seria idêntico — mas `*args` e `**kwargs` são tão difundidos que se tornaram o padrão da comunidade.

**`*args`: múltiplos argumentos posicionais**

- O `*args` permite que uma função receba vários argumentos posicionais, agrupando-os em uma tupla.

In [ ]:
def adicao(*args):
    resultado = 0

    for argumento in args:
        resultado += argumento

    return resultado

print(adicao(1, 2))
print(adicao(1, 2, 3, 4))
print(adicao(1, 2, 3, 4, 5, 6))

### 📊 Exemplo Econômico: Calculando o PIB com `*args`

A composição do PIB pode variar de país para país — alguns modelos incluem mais ou menos componentes.

$$Y = C + I + G + (X - M)$$

In [ ]:
def calcular_pib(consumo, investimento, gastos_governo, exportacoes, importacoes):
    pib = consumo + investimento + gastos_governo + (exportacoes - importacoes)
    print(f"PIB calculado: R$ {pib:.2f} bilhões")

calcular_pib(
    consumo = 4800,
    investimento = 1100,
    gastos_governo = 2200,
    exportacoes = 1600,
    importacoes = 1400
)

Com `*args`, podemos criar uma função que soma todos os componentes do PIB, independentemente de quantos forem passados:

In [ ]:
def calcular_pib(*componentes):
    """
    Soma todos os componentes do PIB passados como argumentos.
    Aceita qualquer quantidade de componentes.
    """
    return sum(componentes)


# PIB simplificado (apenas C e G)
pib_simples = calcular_pib(4800, 2200, 1600)
print(f"PIB simplificado: R$ {pib_simples:,.0f} bi")

# PIB completo (C + I + G + X - M)
pib_completo = calcular_pib(4800, 1100, 2200, 1600, -1400)
print(f"PIB completo:     R$ {pib_completo:,.0f} bi")

**`**kwargs`: múltiplos argumentos nomeados**

- O `**kwargs` permite passar argumentos com nomes (chave=valor), agrupando-os em um dicionário (`{'chave': 'valor'}`) dentro da função.

In [ ]:
def concatenar(**kwargs):
    print(f'Valores recebidos: {kwargs}')
    resultado = ""

    for valor in kwargs.values():
        resultado += f'{valor} '
    return resultado


print(concatenar(curso="Economia"))
print()
print(concatenar(curso="Economia", faculdade="FEA-USP"))


### 📊 Exemplo Econômico: Relatório de indicadores macroeconômicos com `**kwargs`

Com `**kwargs`, podemos construir uma função que gera relatórios de indicadores macroeconômicos com qualquer conjunto de variáveis — útil quando comparamos países ou períodos com disponibilidades de dados diferentes:

In [ ]:
def relatorio_macro(pais, **indicadores):
    print(f"\n=== Indicadores Macroeconômicos: {pais} ===")
    for indicador, valor in indicadores.items():
        # Substitui underscores por espaços para melhor legibilidade
        nome_formatado = indicador.replace("_", " ").title()
        print(f"  {nome_formatado}: {valor}")

relatorio_macro(
    "Brasil",
    pib_crescimento="2.9%",
    inflacao_ipca="4.83%",
    taxa_desemprego="6.2%",
    taxa_selic="10.5%"
)

relatorio_macro(
    "Argentina",
    inflacao="211%",
    taxa_desemprego="7.7%"
    # Menos indicadores disponíveis — sem problema!
)

**Usando `*args` e `**kwargs` juntos**

- Também é possível combinar os dois:

In [ ]:
def relatorio_economico(*args, **kwargs):
    print("Valores agregados:", sum(args))
    for chave, valor in kwargs.items():
        print(f"{chave}: {valor}")

relatorio_economico(100, 200, consumo=5000, investimento=2000)

## Funções anônimas (lambda) <a id="funcoes-anonimas"></a>

Em Python, funções lambda são funções anônimas, ou seja, funções sem nome. Elas são definidas usando a palavra-chave `lambda`, que é seguida por parâmetros (opcionais) e uma expressão.

A expressão é avaliada e retorna o resultado quando a função `lambda` é chamada. A sintaxe básica de uma função lambda é:

```python
lambda <argumentos> : <expressão>
```

In [ ]:
soma = lambda x, y: x + y # Função lambda atribuída a uma variável
print(soma(5, 3))

(lambda x, y: x + y)(5, 3) # Sem atribuir uma função lambda a uma variável

- Neste exemplo, `lambda x, y: x + y` define uma função lambda que recebe dois argumentos x e y e retorna sua soma. Essa função lambda é atribuída a uma variável chamada `soma`, e então chamamos `soma(3, 5)` para obter o resultado da adição de 3 e 5.
- Note que, para chamar uma função lambda, usamos a variável à qual atribuímos a expressão lambda.
- Neste quesito, a função é chamada exatamente da mesma forma que uma função "tradicional", definida com `def`.

**Para que serve uma função lambda em Python?**

- As funções lambda são frequentemente usadas em situações onde você precisa de uma função temporária para uma única expressão simples.
- Essas funções são frequentemente criadas para servir de argumentos a outras funções, sem que sejam armazenadas em variáveis.
- Por exemplo: operações de filtragem (`filter`), mapeamento (`map`) e ordenação (`sorted`).

### Função `map()`

- A função `map()` pode ser utilizada junto a uma função lambda para aplicar a operação a cada item de uma lista:

In [ ]:
# Elevar ao quadrado cada número em uma lista
numeros = [1, 2, 3, 4, 5]
resultado = map(lambda x: x ** 2, numeros)

print(list(resultado))

### 📊 Exemplo Econômico: Corrigindo preços pela inflação com `map()`

Em economia, **deflacionar** uma série de preços significa remover o efeito da inflação para tornar valores de diferentes períodos comparáveis em termos reais. A fórmula geral é:

$$P_{\text{real}} = \frac{P_{\text{nominal}}}{1 + \pi}$$

onde $P_{\text{nominal}}$ é o preço observado no período e $\pi$ é a taxa de inflação acumulada entre o período de referência e o período de base. O denominador $(1 + \pi)$ atua como um **deflator**: quanto maior a inflação, mais o preço nominal é "encolhido" para revelar seu valor real.

Suponha que temos uma lista de preços de um produto ao longo do tempo e queremos corrigi-los pela inflação acumulada de 15%:

In [ ]:
precos_nominais = [10.00, 12.50, 9.80, 15.00, 11.30]
inflacao_acumulada = 0.15  # 15% de inflação acumulada no período

precos_reais = list(map(lambda p: p / (1 + inflacao_acumulada), precos_nominais))

print("Preço nominal → Preço real (deflacionado)")
for nominal, real in zip(precos_nominais, precos_reais):
    print(f"  R$ {nominal:.2f}  →  R$ {real:.2f}")

### Função `filter()`

- De forma semelhante, a função `filter()` pode ser usada para filtrar elementos de uma lista que atendam a um critério de verdadeiro/falso definido pela função lambda:

In [ ]:
# Filtrar números pares de uma lista
numeros = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
numeros_pares = list(filter(lambda x: x % 2 == 0, numeros))

print(numeros_pares)

### 📊 Exemplo Econômico: Filtrando países com alta inflação

Com `filter()`, podemos identificar facilmente quais países de uma lista têm indicadores acima de um determinado limiar — por exemplo, quais têm inflação superior à meta:

In [ ]:
# Lista de tuplas: (país, inflação anual em %)
paises_inflacao = [
    ("Brasil", 4.8),
    ("Chile", 3.2),
    ("Argentina", 211.0),
    ("México", 4.7),
    ("Colômbia", 9.3),
    ("Peru", 2.9),
    ("Uruguai", 5.1),
]

meta_inflacao = 4.5  # meta hipotética comum

# Filtra países com inflação acima da meta
acima_da_meta = list(filter(lambda x: x[1] > meta_inflacao, paises_inflacao))

print(f"Países com inflação acima de {meta_inflacao}%:")
for pais, inf in acima_da_meta:
    print(f"  {pais}: {inf}%")

### Função `sorted()`

A função `sorted()` ordena sequências. Ela aceita o parâmetro `key`, que é uma função contendo a lógica de ordenamento a ser usada.

No exemplo a seguir, a função `sorted` utiliza a lógica descrita pela função lambda para ordenar os elementos de `minha_lista`:

- A função lambda toma cada tupla (x) como argumento e retorna o segundo elemento da tupla (`x[1]`). Assim, a lista de tuplas é ordenada pelo número de forma ascendente.

Dicionários em Python armazenam pares `chave:valor`. O método `.items()` retorna esses pares como uma sequência de tuplas `(chave, valor)`, o que permite iterá-los e ordená-los como qualquer outra sequência.

Na função lambda abaixo, a variável x representa cada tupla de par chave-valor do dicionário, e a lógica `x[1]` indica que o segundo elemento dessa tupla (o valor de cada par chave-valor) deve ser usada para o ordenamento.

In [ ]:
minha_lista = [('maçã', 1), ('banana', 3), ('laranja', 2)]
minha_lista_ordenada = sorted(minha_lista, key = lambda x: x[1])
print(minha_lista_ordenada)

# Função lambda para ordenar um dicionário
notas = {'Danilo': 7, 'Mariana': 9, 'Bruno': 4, 'Daniela': 8.5}
notas_ordenadas = sorted(
    notas.items(),
    key = lambda x: x[1],
    reverse = True
)
print(notas_ordenadas)

### 📊 Exemplo Econômico: Ranking de países por PIB per capita

Uma aplicação clássica de `sorted()` em economia é ordenar países por algum indicador — como o PIB per capita — para compará-los:

In [ ]:
# (país, PIB per capita em US$ mil, taxa de desemprego em %)
paises = [
    ("Brasil",    9.0,  6.2),
    ("Argentina", 11.6, 7.7),
    ("Chile",     16.5, 8.9),
    ("México",    10.8, 2.7),
    ("Colômbia",   6.8, 9.1),
    ("Peru",       6.6, 7.4),
]

# Ordenando por PIB per capita (maior para menor)
ranking_pib = sorted(paises, key=lambda x: x[1], reverse=True)
print("Ranking por PIB per capita (US$ mil):")
for i, (pais, pib, _) in enumerate(ranking_pib, start=1):
    print(f"  {i}º {pais}: US$ {pib:.1f} mil")

print()

# Ordenando por taxa de desemprego (menor para maior)
ranking_desemp = sorted(paises, key=lambda x: x[2])
print("Ranking por taxa de desemprego (menor → maior):")
for i, (pais, _, desemp) in enumerate(ranking_desemp, start=1):
    print(f"  {i}º {pais}: {desemp}%")

> 💡 **Sobre a função `enumerate()`**
> A função `enumerate()` permite percorrer uma lista obtendo ao mesmo tempo o índice e o valor de cada elemento. No exemplo, ela é usada para gerar automaticamente a posição no ranking (1º, 2º, etc.), começando em 1 com o parâmetro `start=1`, sem precisar controlar manualmente um contador.